In [28]:
import os
import numpy as np
import glob
from PIL import Image, ImageDraw

# pip install torchsummary
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import models
from torchsummary import summary
import torch.optim as optim
from time import time

import matplotlib.pyplot as plt
from IPython.display import clear_output
import random
from scipy.ndimage import measurements
from skimage import measure
from scipy.ndimage import distance_transform_edt
from glob import glob
import os
import pandas as pd
from torchvision import transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [5]:
import sys
sys.path.append('/zhome/a4/f/202251/projects/02516-IDLCV/')

In [ ]:
class FrameImageDataset(torch.utils.data.Dataset):
    def __init__(self,
    root_dir='/work3/ppar/data/ucf101',
    split='train',
    transform=None
):
        self.frame_paths = sorted(glob(f'{root_dir}/frames/{split}/*/*/*.jpg'))
        self.df = pd.read_csv(f'{root_dir}/metadata/{split}.csv')
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.frame_paths)

    def _get_meta(self, attr, value):
        return self.df.loc[self.df[attr] == value]

    def __getitem__(self, idx):
        frame_path = self.frame_paths[idx]
        video_name = frame_path.split('/')[-2]
        video_meta = self._get_meta('video_name', video_name)
        label = video_meta['label'].item()

        frame = Image.open(frame_path).convert("RGB")

        if self.transform:
            frame = self.transform(frame)
        else:
            frame = T.ToTensor()(frame)

        return frame, label


class FrameVideoDataset(torch.utils.data.Dataset):
    def __init__(self,
    root_dir = '/work3/ppar/data/ucf101',
    split = 'train',
    transform = None,
    stack_frames = True
):

        self.video_paths = sorted(glob(f'{root_dir}/videos/{split}/*/*.avi'))
        self.df = pd.read_csv(f'{root_dir}/metadata/{split}.csv')
        self.split = split
        self.transform = transform
        self.stack_frames = stack_frames

        self.n_sampled_frames = 10

    def __len__(self):
        return len(self.video_paths)

    def _get_meta(self, attr, value):
        return self.df.loc[self.df[attr] == value]

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        video_name = video_path.split('/')[-1].split('.avi')[0]
        video_meta = self._get_meta('video_name', video_name)
        label = video_meta['label'].item()

        video_frames_dir = self.video_paths[idx].split('.avi')[0].replace('videos', 'frames')
        video_frames = self.load_frames(video_frames_dir)

        if self.transform:
            frames = [self.transform(frame) for frame in video_frames]
        else:
            frames = [T.ToTensor()(frame) for frame in video_frames]

        if self.stack_frames:
            frames = torch.stack(frames).permute(1, 0, 2, 3)


        return frames, label

    def load_frames(self, frames_dir):
        frames = []
        for i in range(1, self.n_sampled_frames + 1):
            frame_file = os.path.join(frames_dir, f"frame_{i}.jpg")
            frame = Image.open(frame_file).convert("RGB")
            frames.append(frame)

        return frames

In [ ]:
print(os.getcwd())
root_dir = '/zhome/a4/f/202251/projects/02516-IDLCV/data/ufc10'

transform = T.Compose([T.Resize((64, 64)),T.ToTensor()])
frameimage_train_dataset = FrameImageDataset(root_dir=root_dir, split='train', transform=transform)
framevideostack_train_dataset = FrameVideoDataset(root_dir=root_dir, split='train', transform=transform, stack_frames = True)
framevideolist_train_dataset = FrameVideoDataset(root_dir=root_dir, split='train', transform=transform, stack_frames = False)

frameimage_val_dataset = FrameImageDataset(root_dir=root_dir, split='val', transform=transform)
framevideostack_val_dataset = FrameVideoDataset(root_dir=root_dir, split='val', transform=transform, stack_frames = True)
framevideolist_val_dataset = FrameVideoDataset(root_dir=root_dir, split='val', transform=transform, stack_frames = False)

frameimage_test_dataset = FrameImageDataset(root_dir=root_dir, split='test', transform=transform)
framevideostack_test_dataset = FrameVideoDataset(root_dir=root_dir, split='test', transform=transform, stack_frames = True)
framevideolist_test_dataset = FrameVideoDataset(root_dir=root_dir, split='test', transform=transform, stack_frames = False)

frameimage_train_loader = DataLoader(frameimage_train_dataset,  batch_size=8, shuffle=False)
framevideostack_train_loader = DataLoader(framevideostack_train_dataset,  batch_size=8, shuffle=False)
framevideolist_train_loader = DataLoader(framevideolist_train_dataset,  batch_size=8, shuffle=False)

frameimage_val_loader = DataLoader(frameimage_val_dataset,  batch_size=8, shuffle=False)
framevideostack_val_loader = DataLoader(framevideostack_val_dataset,  batch_size=8, shuffle=False)
framevideolist_val_loader = DataLoader(framevideolist_val_dataset,  batch_size=8, shuffle=False)

frameimage_test_loader = DataLoader(frameimage_test_dataset,  batch_size=8, shuffle=False)
framevideostack_test_loader = DataLoader(framevideostack_test_dataset,  batch_size=8, shuffle=False)
framevideolist_test_loader = DataLoader(framevideolist_test_dataset,  batch_size=8, shuffle=False)

# for frames, labels in frameimage_loader:
#     print(frames.shape, labels.shape) # [batch, channels, height, width]

# for video_frames, labels in framevideolist_loader:
#     print(45*'-')
#     for frame in video_frames: # loop through number of frames
#         print(frame.shape, labels.shape)# [batch, channels, height, width]

# for video_frames, labels in framevideostack_loader:
#     print(video_frames.shape, labels.shape) # [batch, channels, number of frames, height, width]

/zhome/a4/f/202251


[0 1 2 3 4 5 6 7 8 9]


In [ ]:
class CNNNet(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.num_classes = num_classes
    self.cnn_layer = nn.Sequential(
      # Block 1
      nn.Conv2d(3, 64, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.Conv2d(64, 64, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),

      # Block 2
      nn.Conv2d(64, 128, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.Conv2d(128, 128, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),

      # Block 3
      nn.Conv2d(128, 256, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.Conv2d(256, 256, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.Conv2d(256, 256, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
    )

    self.flatten = nn.Flatten()

    self.fc_layer = nn.Sequential(
      nn.Linear(256*8*8, 256),
      nn.ReLU(),
      nn.Dropout(0.5),
      nn.Linear(256, 256),
      nn.ReLU(),
      nn.Dropout(0.5),
      nn.Linear(256, self.num_classes)
    )

  def forward(self, images):
    x = self.cnn_layer(images)
    x = self.flatten(x)
    x = self.fc_layer(x)

    return x


class AverageNet(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.num_classes = num_classes
    self.cnn_predictor = CNNNet(self.num_classes).to(device)

  def forward(self, images):
    prediction = self.cnn_predictor(images)
    print(prediction.shape)
    average = torch.mean(prediction, dim=1)

    return average

In [49]:
def evaluate_model(model, dataloader):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for video_frames, labels in dataloader:
            video_frames, labels = video_frames.to(device), labels.to(device)
            outputs = model(video_frames)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    print(f"Validation Accuracy: {accuracy:.4f}")

def train(model, train_loader, val_loader, criterion, optimizer, num_epochs = 30):

  model.train()
  for epoch in range(num_epochs):
    epoch_loss = 0;
    correct = 0;
    total = 0;

    for video_frames, labels in train_loader:
      video_frames, labels = video_frames.to(device), labels.to(device).float()

      outputs = model(video_frames)
      print(outputs.shape, labels.shape)
      loss = criterion(outputs, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      # Track loss and accuracy
      epoch_loss += loss.item()
      print(outputs.shape)
      predicted = torch.max(outputs.data, 1)
      print(predicted.shape)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

      # val
      evaluate_model(model, val_loader)
      model.train()

    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

averageModel = AverageNet(10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(averageModel.parameters())
train(averageModel, frameimage_train_loader, frameimage_val_loader, criterion, optimizer)

torch.Size([8]) torch.Size([8])
torch.Size([8])


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)